In [1]:
# For processing the timeseries
import pandas as pd, os, datetime
import numpy as np

# PLotting
import plotly.graph_objects as go

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA/case_studies/coal_participation')

geninfo_path = '/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess'
ehf_fpath = '/scratch/ng72/ms5578/time_series'
gen_fpath = '/scratch/ng72/ms5578/time_series/nem_generation'

In [3]:
hw_tseries = pd.read_csv(f"{ehf_fpath}/gen_hw_status.csv")

In [4]:
gen_info = pd.read_csv(f"{geninfo_path}/gen_info.csv")

In [5]:
def process_group(grp, gen_fpath, hw_tseries, start_date=None, end_date=None):
    """
    Processes a single group of generator data.

    Parameters:
        grp (pd.DataFrame): A single group from gen_info (e.g., from groupby('region')).
        gen_fpath (str): Path to directory containing CSV files named by DUID (e.g., DUID.csv).
        hw_tseries (pd.DataFrame): DataFrame with columns 'time', 'DUID', and data to merge.
        start_date (str): Start date for subsetting the time series.
        end_date (str): End date for subsetting the time series.

    Returns:
        pd.DataFrame: The merged result for the group.
    """
    safe_duids = [duid.replace("/", "_").replace("\\", "_") for duid in grp['DUID']]
    gen_locs = [f"{gen_fpath}/{duid}.csv" for duid in safe_duids]
    dfs = [pd.read_csv(fp,dtype='object') for fp in gen_locs if os.path.exists(fp)]
    print(f"Loaded {len(dfs)} CSV file(s) out of {len(gen_locs)} expected.")

    if not dfs:
        return None

    # This is in case of accidental mid-file headers
    dfs = pd.concat(dfs, ignore_index=True)
    header_row = dfs.columns.tolist()
    dfs = dfs[~dfs.apply(lambda row: list(row) == header_row, axis=1)]

    # This is to correct the column types after removing header rows
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs['TOTALCLEARED'] = dfs['TOTALCLEARED'].astype(float)
    dfs['TOTALMWh'] = dfs['TOTALMWh'].astype(float)
    dfs['AGCSTATUS'] = pd.to_numeric(dfs['AGCSTATUS'], errors='coerce').fillna(0).astype(int)
    
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs = dfs.set_index('time').sort_index()
    dfs = dfs.loc[start_date:end_date]

    # Group by DUID and hourly time, sum TOTALMWh
    agg_func = {'TOTALMWh':'sum','TOTALCLEARED':'sum','AGCSTATUS':'max'}
    agg_df = dfs.groupby(['DUID', pd.Grouper(freq='1d')]).agg(agg_func).reset_index()

    hw_tseries['time'] = pd.to_datetime(hw_tseries['time'])
    hw_tseries = hw_tseries.set_index(['time']).sort_index()
    hw_tseries = hw_tseries.loc[sdate:edate]
    hw_tseries = hw_tseries.reset_index().set_index(['DUID','time']).sort_index()

    merged = pd.merge_asof(
        agg_df.sort_values(by=['time', 'DUID']),
        hw_tseries.sort_values(by=['time', 'DUID']),
        by='DUID',
        on='time',
        tolerance=pd.Timedelta("1d"),
        direction='nearest'
    )
    
    return merged

In [6]:
def select_group(gen_info=gen_info, state=None,ftype=None):
    if state is not None and ftype is not None:
        groups = gen_info.groupby(['region','fuel_source_primary'])
        grp = groups.get_group((state,ftype))
    elif state is not None:
        groups = gen_info.groupby('region')
        grp = pd.concat([groups.get_group(i) for i in state])
    elif ftype is not None:
        groups = gen_info.groupby('fuel_source_primary')
        grp = pd.concat([groups.get_group(i) for i in ftype])
    else:
        grp = gen_info

    return grp


In [7]:
sdate, edate = '2009-07-01','2024-06-30'
group = select_group(ftype=['Black Coal','Brown Coal']).copy()
df = process_group(group, gen_fpath, hw_tseries,sdate,edate)
djf = [12, 1, 2]  # December, January, February
df = df[df['time'].dt.month.isin(djf)]

Loaded 44 CSV file(s) out of 48 expected.


In [10]:
yearly = df.pivot_table(
    index=['DUID', 'EHF_flag'],
    columns=pd.Grouper(key='time', freq='YE'),
    values='TOTALMWh',
    aggfunc='mean'
    )

yearly_long = yearly.reset_index().melt(
    id_vars=['DUID','EHF_flag'],
    var_name='year',
    value_name='mean_TOTALMWh'
    )
yearly_long['year'] = pd.to_datetime(yearly_long['year']).dt.year.astype(int)

In [11]:
yearly_long

,DUID,EHF_flag,year,mean_TOTALMWh
0,BW01,0.0,2009,10439.419950
1,BW01,1.0,2009,6228.204190
2,BW02,0.0,2009,11566.534587
3,BW02,1.0,2009,8756.940867
4,BW03,0.0,2009,11854.046686
...,...,...,...,...
1403,YWPS2,1.0,2024,NaN
1404,YWPS3,0.0,2024,7762.258274
1405,YWPS3,1.0,2024,NaN
1406,YWPS4,0.0,2024,4812.364150


In [14]:
dist_df = df[['DUID','EHF_flag','TOTALMWh','time']].copy()
dist_df['time'] = pd.to_datetime(dist_df['time'])
dist_df['year'] = dist_df['time'].dt.year

In [19]:
count_table = (
    dist_df[dist_df['EHF_flag'] == 1]
      .groupby(['DUID', 'year'])
      .size()
      .reset_index(name='EHF_flag_1_days')
)

print(count_table)

      DUID  year  EHF_flag_1_days
0     BW01  2009                6
1     BW01  2010               15
2     BW01  2011               10
3     BW01  2012                3
4     BW01  2013               16
..     ...   ...              ...
599  YWPS4  2018               22
600  YWPS4  2019               17
601  YWPS4  2020                5
602  YWPS4  2021                6
603  YWPS4  2022                9

[604 rows x 3 columns]


In [18]:
from scipy.stats import mannwhitneyu

# Store results
results = []

# Loop through each year
for year, group in dist_df.groupby(['DUID','year']):
    group0 = group[group['EHF_flag'] == 0]['TOTALMWh']
    group1 = group[group['EHF_flag'] == 1]['TOTALMWh']
    
    if len(group0) > 0 and len(group1) > 0:
        stat, p = mannwhitneyu(group0, group1, alternative='two-sided')
        results.append({'year': year, 'U_stat': stat, 'p_value': p})
    else:
        results.append({'year': year, 'U_stat': None, 'p_value': None})

# Convert to DataFrame for easy viewing
mannwhitney_results = pd.DataFrame(results)
mannwhitney_results


,year,U_stat,p_value
0,"(BW01, 2009)",106.0,0.126492
1,"(BW01, 2010)",376.0,0.044035
2,"(BW01, 2011)",86.0,0.000057
3,"(BW01, 2012)",209.0,0.091452
4,"(BW01, 2013)",583.5,0.931894
...,...,...,...
694,"(YWPS4, 2020)",52.0,0.004650
695,"(YWPS4, 2021)",135.0,0.059486
696,"(YWPS4, 2022)",306.0,0.431486
697,"(YWPS4, 2023)",NaN,NaN
